# The German Lab: content and progress notebook

Use this notebook **next to** the website, not instead of it. It reads the same database, so you can:

1. check the content (how many words, questions and tasks per level),
2. see how your students are doing, as a table,
3. import many words at once from a CSV file.

**Start it from the project folder** (the one with `manage.py`), with the project's environment active:

```
.venv\Scripts\activate
pip install -r requirements-dev.txt
jupyter notebook notebooks/german_lab.ipynb
```

## 1. Connect to the website's database

In [ ]:
import os, sys
from pathlib import Path

# Find the project folder (works whether Jupyter started in notebooks/ or the project root)
ROOT = Path.cwd() if (Path.cwd() / "manage.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT))
os.chdir(ROOT)
os.environ.setdefault("DJANGO_SETTINGS_MODULE", "config.settings.dev")
os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"  # Jupyter runs an event loop; this lets Django queries run in cells

import django
django.setup()

import pandas as pd
from django.db.models import Count, Q
from accounts.models import User, AccessRequest
from content.models import Level, Topic, VocabularyItem, QuizQuestion, SpeakingTask, GrammarTopic, Video
from learning.models import CardReview, QuizAttempt
from learning.services import student_summary
print("Connected to", django.conf.settings.DATABASES["default"]["NAME"])

## 2. Content per level

In [ ]:
rows = []
for level in Level.objects.all():
    rows.append({
        "level": level.code,
        "word decks": level.topics.count(),
        "words": VocabularyItem.objects.filter(topic__level=level).count(),
        "quiz questions": level.questions.count(),
        "speaking tasks": level.speaking_tasks.count(),
        "grammar topics": level.grammar_topics.count(),
    })
pd.DataFrame(rows).set_index("level")

Quiz questions per grammar topic. Topics with few questions are good candidates for new ones.

In [ ]:
pd.DataFrame(
    QuizQuestion.objects.values("level__code", "grammar_topic__title").annotate(questions=Count("id"))
    .order_by("level__code", "-questions")
).rename(columns={"level__code": "level", "grammar_topic__title": "grammar topic (empty = vocabulary)"})

## 3. How your students are doing

In [ ]:
students = User.objects.filter(role="student", access_request__status="approved").select_related("classroom")
table = pd.DataFrame([
    {"student": s.full_name, "class": str(s.classroom or "-"), **student_summary(s)}
    for s in students
])
table if not table.empty else "No approved students yet." 

The hardest words: the ones students most often answer *Not yet*.

In [ ]:
df = pd.DataFrame(CardReview.objects.values("item__article", "item__german", "item__english", "times_seen", "times_correct"))
if df.empty:
    print("No flashcard reviews yet.")
else:
    df["wrong"] = df.times_seen - df.times_correct
    df["word"] = (df.item__article + " " + df.item__german).str.strip()
    display(df.groupby(["word", "item__english"])[["times_seen", "wrong"]].sum()
              .assign(error_rate=lambda d: (d.wrong / d.times_seen).round(2))
              .sort_values("error_rate", ascending=False).head(15))

## 4. Import words from a CSV file

Make a CSV (e.g. in Excel, *Save as CSV UTF-8*) with these columns:

| level | topic | article | german | english | example |
|---|---|---|---|---|---|
| A1 | Kleidung | die | Hose | the trousers | Die Hose ist zu lang. |

`article` can be `der`, `die`, `das` or empty. New topics are created automatically. Run with `DRY_RUN = True` first to check.

In [ ]:
CSV_FILE = ROOT / "notebooks" / "new_words.csv"   # change to your file
DRY_RUN = True                                      # set to False to really save

from django.db import transaction

def import_words(path, dry_run=True):
    data = pd.read_csv(path, dtype=str).fillna("")
    problems, created, updated = [], 0, 0
    with transaction.atomic():
        for i, row in data.iterrows():
            line = i + 2  # row number as Excel shows it
            level = Level.objects.filter(code=row["level"].strip().upper()).first()
            if not level:
                problems.append(f"Row {line}: unknown level '{row['level']}'"); continue
            if row["article"].strip() not in {"", "der", "die", "das"}:
                problems.append(f"Row {line}: article must be der, die, das or empty"); continue
            if not row["german"].strip() or not row["english"].strip():
                problems.append(f"Row {line}: german and english are required"); continue
            topic, _ = Topic.objects.get_or_create(level=level, name=row["topic"].strip() or "Allgemein")
            _, was_created = VocabularyItem.objects.update_or_create(
                topic=topic, article=row["article"].strip(), german=row["german"].strip(),
                defaults={"english": row["english"].strip(), "example": row["example"].strip()},
            )
            created += was_created; updated += not was_created
        if dry_run:
            transaction.set_rollback(True)
    print(f"{'Would create' if dry_run else 'Created'} {created}, {'would update' if dry_run else 'updated'} {updated}.")
    for p in problems:
        print("  -", p)

if CSV_FILE.exists():
    import_words(CSV_FILE, DRY_RUN)
else:
    print(f"Put your CSV at {CSV_FILE} (an example file is included).")